# Min Spanning Tree with One Free Edge

Given an undirected weighted graph with n nodes (0 to n-1) and m edges, compute the
minimum possible weight of a spanning tree if you may choose at most one edge and set its
weight to zero.

## Problem
You are given `n`, `m` and a list of `m` edges where each edge is described by three integers
`u v w` — an undirected edge between nodes `u` and `v` with weight `w`.
You may pick at most one edge and treat its weight as 0 before building the spanning tree.
Return the minimum total weight of a spanning tree achievable under this rule.

## Input Format
The first line contains two space-separated integers `n` and `m`.
Each of the next `m` lines contains three space-separated integers `u v w`.

## Constraints
- 1 <= n <= 100000
- n-1 <= m <= min(200000, n*(n-1)/2)
- 0 <= u, v < n; u != v
- 0 <= w <= 10^9
- The graph is connected.

## Output Format
Print a single 64-bit integer: the minimum possible sum of weights of any spanning tree after
optionally setting one edge's weight to zero.

## Sample Input
```
3 3
0 1 3
1 2 4
0 2 5
```
## Sample Output
```
3
```

## Explanation
Without a free edge, the MST cost is 3 + 4 = 7. If we set edge (1,2) to zero, we can take
edges with weights 0 and 3 for total 3, which is minimal.

## Notes
- Multiple edges between the same pair of nodes are allowed and should be treated independently.
- Use 64-bit integers for accumulated sums to avoid overflow.




1) ESPECIFICAÇÃO:

- Núcleo: Calcular o peso mínimo de uma árvore geradora (MST) em um grafo não direcionado e conexo,
  permitindo definir o peso de no máximo uma aresta como zero; escolher a aresta que minimiza o peso final.
- Blueprint: Construir a MST com Kruskal e usar lifting binário (LCA) para consultar a aresta de maior peso
  no caminho entre dois nós na MST; avaliar o efeito de zerar cada aresta original.
Blueprint (esqueleto tipado):
```python
from typing import List, Tuple

class Solution:
    def min_spanning_with_one_free_edge(self, n: int, edges: List[Tuple[int, int, int]]) -> int:
        """Retorna o peso mínimo da MST após, opcionalmente, definir uma aresta como zero."""
        ...
```
Casos de borda e pegadinhas:

- pesos muito grandes: use inteiros de 64 bits para acumular somas (overflow).
- arestas paralelas: trate cada aresta independentemente; a MST pode escolher apenas uma delas.
- pesos idênticos: empates em Kruskal não devem comprometer a correção do algoritmo.
- zerar aresta já na MST vs aresta fora da MST: se a aresta está na MST, o ganho é seu próprio peso;
  caso contrário, o ganho é igual à aresta de maior peso no caminho entre seus extremos na MST.
- tamanhos máximos (n ≤ 1e5, m ≤ 2e5): evitar varreduras O(n*m) ou reconstruções repetidas da MST.

2) PLANO (Heurística e Desenho):
- Ponte da intuição (força bruta): testar cada aresta tornando-a zero e recomputar a MST — O(m * (m log m)) é inviável em escala.
- Projeto otimizado:
  1) Executar Kruskal uma vez para obter uma MST (custo = mst_weight) e marcar arestas usadas.  # O(m log m)
  2) Construir a adjacência da MST (n-1 arestas) e pré-processar LCA com lifting binário para responder
     consultas 'maior aresta no caminho(u,v)' em O(log n).  # O(n log n) pré-processamento
  3) Para cada aresta original (u,v,w):
     - se está na MST: candidato = mst_weight - w (zerar essa aresta).
     - senão: max_no_caminho = query(u,v); candidato = mst_weight - max_no_caminho (substituímos a aresta mais pesada).
  4) Resposta = mínimo entre todos os candidatos (inclui mst_weight quando não usar aresta gratuita).
Traço visual (exemplo pequeno):
- Nós: 0,1,2. Arestas: (0,1,3), (1,2,4), (0,2,5).
  Kruskal escolhe (0,1)=3 e (1,2)=4 → mst_weight = 7.
  Consulta maior aresta no caminho(0,2) na MST → max = 4. Se tornarmos (0,2) gratuita, novo custo = 7 - 4 = 3.

3) IMPLEMENTAÇÃO (Legível e anotada):
- Use DSU (Union-Find com compressão de caminho) para Kruskal.
- Pré-processe tabelas de lifting (`up`) e `max_up` armazenando 2^k ancestrais e a maior aresta até esse ancestral.

- Comentários de complexidade são adicionados inline no código de implementação.
4) COMPLEXIDADE (Veredito):
- Tempo: ordenação O(m log m) + pré-processamento O(n log n) + consultas O(m log n) ⇒ O((m + n) log n).
- Espaço: O(n log n + m) para tabelas de LCA e listas de arestas.

---

**Explicação didática: Binary Lifting para 'max edge on path'**

Objetivo: dado o grafo MST (árvore) queremos responder 'qual é a maior aresta no caminho entre a e b' em O(log n).

Estratégia resumida:
- Para cada nó `v` e cada k>=0, guardamos `up[v][k]` = ancestral de `v` a 2^k passos acima,
  e `max_up[v][k]` = maior peso de aresta no caminho de `v` até `up[v][k]`.
- Com essas tabelas podemos subir `v` e `u` em saltos de potências de dois, agregando o máximo visto,
  até que ambos alcancem o mesmo ancestral (o LCA).

Exemplo didático (tabela simplificada):
Suponha árvore com arestas (v -> parent, peso):
- 5 <-2(3), 2<-0(1), 3<-1(4) (exemplo hipotético)

Calcule `up` e `max_up` para k=0..2:
- `up[v][0]` = parent imediato; `max_up[v][0]` = peso da aresta até o parent.
- `up[v][1]` = `up[ up[v][0] ][0]` (ancestral a 2 passos); `max_up[v][1]` = max(max_up[v][0], max_up[ up[v][0] ][0]).

Como responder query(a,b):
1) Igualar profundidades: suba o nó mais profundo com saltos binários somando máximos vistos. (O(log n))
2) Se iguais já termine: o máximo agregado é resposta. Senão, suba ambos em potências de dois, do maior k ao menor,
   sempre que seus `up` em nível k forem diferentes; atualize o máximo com `max_up` nesses saltos. (O(log n))
3) Por fim, considere os saltos de um nível (k=0) para incluir as arestas que chegam ao LCA.

Por que é O(log n)? Porque cada salto reduz a diferença de profundidade em potências de 2, e o loop principal varre k de LOG..0.

Dica visual: para subir `a` em 13 (=1101₂) passos, combine saltos 8 + 4 + 1, ou seja, use k onde o bit está 1.

---


In [ ]:
#!/bin/python3
import math
import os
import random
import re
import sys
from typing import List, Tuple

#
# Complete the 'calculateMinimumSpanningTreeWeightWithFreeEdge' function below.
#
# The function is expected to return a LONG_INTEGER.
# The function accepts following parameters:
#  1. INTEGER n
#  2. INTEGER m
#  3. 2D_INTEGER_ARRAY edges
#

class DSU:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, cell: int) -> int:
        # iterative path compression
        while self.parent[cell] != cell:
            self.parent[cell] = self.parent[self.parent[cell]]
            cell = self.parent[cell]
        return cell

    def union(self, a: int, b: int) -> bool:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True


def calculateMinimumSpanningTreeWeightWithFreeEdge(n: int, m: int, edges: List[List[int]]) -> int:
    """Return minimal MST weight if at most one edge may be set to weight 0.

    Approach:
    1) Build one MST with Kruskal (O(m log m)).
    2) Preprocess MST with binary lifting to answer max-edge-on-path(u,v) in O(log n).
    3) For each original edge, compute candidate = mst_weight - (w if edge in MST else max_on_path(u,v)).
    4) Return min candidate.
    """
    # Sort edges with original indices: O(m log m)
    indexed = [(w, u, v, i) for i, (u, v, w) in enumerate(edges)]
    indexed.sort()

    dsu = DSU(n)
    used = [False] * m
    mst_weight = 0
    adj: List[List[Tuple[int, int]]] = [[] for _ in range(n)]  # MST adjacency

    for w, u, v, idx in indexed:
        if dsu.union(u, v):
            used[idx] = True
            mst_weight += w
            adj[u].append((v, w))
            adj[v].append((u, w))
    # Kruskal done: sorting O(m log m), unions ~ O(m * alpha(n))

    # Binary lifting preprocess: O(n log n) time and space
    # LOG is number of levels (enough to cover height up to n)
    LOG = math.ceil(math.log2(max(2, n))) + 1  # safe for n=1
    up = [[0] * LOG for _ in range(n)]
    max_up = [[0] * LOG for _ in range(n)]
    depth = [0] * n

    # iterative DFS to initialize up[v][0], max_up[v][0], depth[v]
    # We do iterative to avoid recursion depth limits on large n
    stack = [(0, -1, 0)]  # (node, parent, weight_from_parent)
    visited = [False] * n
    while stack:
        node, parent, w_from_parent = stack.pop()
        if visited[node]:
            continue
        visited[node] = True
        # up[node][0] = immediate parent (or itself if root)
        up[node][0] = parent if parent != -1 else node  # O(1)
        # max_up[node][0] = weight of edge (node, parent)
        max_up[node][0] = w_from_parent  # O(1)
        # depth: distance from root in edges
        if parent == -1:
            depth[node] = 0
        else:
            depth[node] = depth[parent] + 1
        for nei, we in adj[node]:
            if not visited[nei]:
                stack.append((nei, node, we))
    # Initialization done: O(n)

    # Fill binary lifting tables: up[v][k] = up[ up[v][k-1] ][k-1]
    # and max_up[v][k] = max(max_up[v][k-1], max_up[ up[v][k-1] ][k-1])
    # Complexity: O(n log n)
    for k in range(1, LOG):
        for v in range(n):
            mid = up[v][k - 1]
            up[v][k] = up[mid][k - 1]
            # max edge from v up to 2^k ancestor is the max of two halves
            max_up[v][k] = max(max_up[v][k - 1], max_up[mid][k - 1])

    def max_edge_on_path(a: int, b: int) -> int:
        """Return maximum edge weight on path a<->b in MST in O(log n).

        Steps:
        1) Bring `a` and `b` to same depth by lifting the deeper node.
        2) If equal, return the aggregated maximum.
        3) Otherwise, lift both simultaneously from high k to low k while their ancestors differ,
           aggregating maximums; finally include the edges to the LCA.
        """
        if a == b:
            return 0
        max_w = 0
        # Ensure a is the deeper node
        if depth[a] < depth[b]:
            a, b = b, a
        # Lift `a` up to depth of `b` using binary representation of the difference
        diff = depth[a] - depth[b]
        k = 0
        while diff:
            if diff & 1:
                # when we jump `a` by 2^k, include the max edge on that jump
                max_w = max(max_w, max_up[a][k])  # O(1)
                a = up[a][k]
            diff >>= 1
            k += 1
        # If after lifting they meet, return collected maximum
        if a == b:
            return max_w
        # Lift both `a` and `b` together from largest k down to 0
        for k in range(LOG - 1, -1, -1):
            if up[a][k] != up[b][k]:
                # include edges of these 2^k jumps for both nodes
                max_w = max(max_w, max_up[a][k], max_up[b][k])
                a = up[a][k]
                b = up[b][k]
        # Finally include the immediate edges to LCA (k=0)
        max_w = max(max_w, max_up[a][0], max_up[b][0])
        return max_w
    # Each query: O(log n)

    ans = mst_weight  # option: do not use free edge
    for i, (u, v, w) in enumerate(edges):
        if used[i]:
            candidate = mst_weight - w  # removing the weight of an MST edge
        else:
            max_on = max_edge_on_path(u, v)  # O(log n)
            candidate = mst_weight - max_on
        if candidate < ans:
            ans = candidate
    return ans


if __name__ == '__main__':
    n = int(input().strip())

    m = int(input().strip())

    edges_rows = int(input().strip())
    edges_columns = int(input().strip())

    edges = []

    for _ in range(edges_rows):
        edges.append(list(map(int, input().rstrip().split())))

    result = calculateMinimumSpanningTreeWeightWithFreeEdge(n, m, edges)

    print(result)
